# FDARP quick start

Runs the pipeline on a single frame (2014-10-20 00:00 UT):

1. download the NOAA SRS file and the Surya-bench magnetogram
2. build and run the C++ detection
3. overlay the detected active regions and the NOAA regions
4. associate detections with NOAA regions

Requirements: `pip install -r ../requirements.txt`, the AWS CLI, and CMake plus
the C++ libraries listed in `../detection/README.md`.

Downloading one frame fetches a ~560 MB netCDF file; only the ~7 MB
magnetogram is kept.

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PIPELINE = ROOT / "pipeline"
DETECTION = ROOT / "detection"
sys.path.insert(0, str(PIPELINE))

import paths
from association import associate_frame, classify
from solar_utils import (C_DETECT, C_LIMB, C_PLAGE, C_SPOT, load_frame,
                         parse_srs)

DATE, TIME = "20141020", "0000"


def run(cmd, cwd):
    """Run a command, stream its output, stop the notebook on failure."""
    print("$", " ".join(str(c) for c in cmd))
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    print(result.stdout[-2000:], result.stderr[-2000:], sep="")
    if result.returncode != 0:
        raise RuntimeError(f"command failed with exit code {result.returncode}")


print("repository:", ROOT)
print("aws CLI   :", shutil.which("aws") or "NOT FOUND")

## 1. Download data

SRS positions are valid at 00:00 UT of the report date.

In [ ]:
run([sys.executable, "download_srs.py", "--date", DATE], cwd=PIPELINE)
run([sys.executable, "download_extract.py", "--start", DATE, "--end", DATE,
     "--daily-at", TIME], cwd=PIPELINE)

for r in parse_srs(paths.srs_file(DATE)):
    kind = "sunspot region" if r["section"] == "I" else "plage"
    print(f"{r['number']}  {kind:<15} lat {r['lat']:>4.0f}  lon {r['lon']:>4.0f}"
          f"  area {r['area'] or '-':>5}  {r['mag'] or ''}")

## 2. Build and run the detection

Builds `fdarp_detect` if needed. On macOS, if linking fails with
`unknown architecture arm64e.x1`, set `SDK` to an installed SDK such as
`/Library/Developer/CommandLineTools/SDKs/MacOSX26.5.sdk`
(see `../detection/README.md`).

The detection processes every magnetogram in `data/fits/<DATE>/` and skips
frames that already have output.

In [ ]:
SDK = None   # e.g. "/Library/Developer/CommandLineTools/SDKs/MacOSX26.5.sdk"

binary = DETECTION / "build" / "fdarp_detect"
if not binary.exists():
    cfg = ["cmake", "-S", ".", "-B", "build"]
    if SDK:
        cfg.append(f"-DCMAKE_OSX_SYSROOT={SDK}")
    run(cfg, cwd=DETECTION)
    run(["cmake", "--build", "build", "-j", "4"], cwd=DETECTION)

run([str(binary), DATE], cwd=DETECTION)

## 3. Detections and NOAA regions

Green: detected active regions. Red: NOAA regions with sunspots. Orange: NOAA
plages. Yellow: the solar limb, with the disk centre and radius measured from
the magnetogram.

In [ ]:
import matplotlib.pyplot as plt

frame = load_frame(DATE, TIME, h5_dir=paths.detection_dir("limb"))
res = associate_frame(frame, parse_srs(paths.srs_file(DATE)), threshold_deg=3.0)

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(frame["mag"], origin="lower", cmap="gray", vmin=-200, vmax=200)
ax.contour(frame["mask"], levels=[0.5], colors=C_DETECT, linewidths=0.8)
for r in res["regions"]:
    col = C_SPOT if r["section"] == "I" else C_PLAGE
    ax.plot(r["x"], r["y"], "o", mfc="none", mec=col, ms=14, mew=1.8)
    ax.text(r["x"] + 40, r["y"] + 40, r["number"], color=col, fontweight="bold")
ax.add_patch(plt.Circle((frame["cx"], frame["cy"]), frame["r_px"],
                        fill=False, color=C_LIMB, ls="--"))
ax.set_title(f"{frame['stamp']}   disk ({frame['cx']:.0f}, {frame['cy']:.0f}), "
             f"R = {frame['r_px']:.0f} px, B0 = {frame['b0']:.2f} deg")
ax.set_xticks([]); ax.set_yticks([])
plt.show()

## 4. Association

A NOAA region is associated with a detected component when the great-circle
distance on the solar surface is at most 3 degrees. Groups:
`1:1`, `1:N` (one component, several regions), `N:1`, `M:N`,
`1:0` (component without a NOAA region), `0:1` (region not detected).

In [ ]:
print("counts:", res["counts"])
for g in res["groups"]:
    comps = ", ".join(f"C{c}" for c in g["components"]) or "-"
    regs = ", ".join(g["regions"]) or "-"
    print(f"  {classify(g):>4}   components [{comps}]   NOAA [{regs}]")

## Next steps

- Hourly frames for a whole day:
  `python download_extract.py --start 20141020 --end 20141020 --hours 1`
- Figures for every frame: `srs_overlay.py`, `plot_association.py`,
  `check_groups.py` in `../pipeline` (written to `../figures/`)
- Many days, one frame per day: `batch_days.py`
- Days with close NOAA region pairs: `scan_srs.py`